# Hotel Booking Cancellation Prediction
## CDS 4003 – Data Science for Business Application

### Stage 3: Data Understanding and Cleaning

This notebook explores the hotel booking dataset,
identifies data quality issues, and prepares the data
for machine learning.

In [1]:
#Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
#loading dataset

df = pd.read_csv("hotel_bookings.csv")

print("Dataset Shape:", df.shape)
display(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'hotel_bookings.csv'

In [ ]:
#Exploring data

# Check column names and data types
df.info()

# Display summary statistics
display(df.describe(include="all"))

# Check missing values
display(df.isnull().sum())

# Check duplicate rows
print("Duplicates:", df.duplicated().sum())

## 6. Data Cleaning

### 6.1 Handling Missing Values

The dataset contains missing values in children, country,
agent, and company. We will handle these values based on
what each column represents.

In [ ]:
#Data Cleaning

# Create a copy to keep the original data unchanged
df_clean = df.copy()

# Fill missing children values with 0
df_clean["children"] = df_clean["children"].fillna(0)

# Fill missing country values with "Unknown"
df_clean["country"] = df_clean["country"].fillna("Unknown")

# Fill missing agent and company IDs with 0
# 0 represents missing or unrecorded IDs
df_clean["agent"] = df_clean["agent"].fillna(0)
df_clean["company"] = df_clean["company"].fillna(0)

# Check missing values after cleaning
print("Missing Values After Cleaning:")
print(df_clean.isnull().sum())

print("Total Missing Values:", df_clean.isnull().sum().sum())


### 6.2 Handling Duplicate Rows

We will check for duplicate rows in the dataset.
Since different hotel bookings can have identical
details, we will investigate them before deciding
whether to remove them.

In [ ]:
# Count duplicate rows
duplicates = df_clean.duplicated().sum()

print("Duplicate Rows:", duplicates)

# Show examples of duplicate rows
display(df_clean[df_clean.duplicated(keep=False)].head(10))

# Check dataset size
print("Total Rows:", len(df_clean))


### 6.3 Checking Invalid Values and Outliers

We will check for invalid and unusual values in the dataset, such as bookings with zero guests, negative hotel prices, and unusually high lead times or daily rates.

These checks will help us identify data quality issues before deciding how to clean them.

In [ ]:
# Check bookings with zero guests
zero_guests = df_clean[
    (df_clean["adults"] +
     df_clean["children"] +
     df_clean["babies"]) == 0
]

print("Bookings with zero guests:", len(zero_guests))

# Check negative hotel prices
print("Negative prices:", (df_clean["adr"] < 0).sum())

# Check unusual lead times
print("Maximum lead time:", df_clean["lead_time"].max())

# Check maximum daily rate
print("Maximum daily rate:", df_clean["adr"].max())

### 6.4 Handling Invalid Values

We will remove bookings with zero guests and negative hotel prices because these values are not valid for normal hotel reservations.

We will keep unusually high lead times and daily rates for further investigation instead of removing them automatically.

In [ ]:
# Store the number of rows before cleaning
rows_before = len(df_clean)

# Remove bookings with zero guests
df_clean = df_clean[
    (df_clean["adults"] +
     df_clean["children"] +
     df_clean["babies"]) > 0
].copy()

# Remove bookings with negative prices
df_clean = df_clean[df_clean["adr"] >= 0].copy()

# Compare before and after
print("Rows Before:", rows_before)
print("Rows After:", len(df_clean))
print("Rows Removed:", rows_before - len(df_clean))

# Verify cleaning
print("Zero Guest Bookings:",
      ((df_clean["adults"] +
        df_clean["children"] +
        df_clean["babies"]) == 0).sum())

print("Negative Prices:",
      (df_clean["adr"] < 0).sum())


### 6.5 Checking Inconsistent Values

We will check the categorical columns for inconsistent values, spelling differences, and unexpected categories.

This will help us decide whether any values need to be standardized.

In [ ]:
# Check unique values in important categorical columns
columns_to_check = [
    "hotel",
    "meal",
    "market_segment",
    "distribution_channel",
    "deposit_type",
    "customer_type",
    "reservation_status"
]

for col in columns_to_check:
    print("\nColumn:", col)
    print(df_clean[col].value_counts(dropna=False))


### 6.6 Data Types and Date Transformation

We will convert the arrival date columns into one date column and check that the data types are correct.

This will make the dataset easier to analyze and prepare for machine learning.

In [ ]:
# Create a complete arrival date
df_clean["arrival_date"] = pd.to_datetime(
    df_clean["arrival_date_year"].astype(str) + "-" +
    df_clean["arrival_date_month"] + "-" +
    df_clean["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d",
    errors="coerce"
)

# Check the new date column
print("Arrival Date Type:", df_clean["arrival_date"].dtype)

# Check for invalid dates
print("Invalid Dates:", df_clean["arrival_date"].isnull().sum())

# Display the first five dates
display(df_clean["arrival_date"].head())


### 6.7 Before and After Data Cleaning

We will compare the original dataset with the cleaned
dataset to show the improvements in data quality.

We will also identify any remaining concerns that
need further investigation.

In [ ]:
# Function to count invalid bookings
def count_invalid(data):
    zero_guests = (
        data["adults"] +
        data["children"].fillna(0) +
        data["babies"]
    ) == 0

    return zero_guests.sum()

# Create before-and-after summary
summary = pd.DataFrame({
    "Quality Check": [
        "Total Rows",
        "Missing Values",
        "Duplicate Rows",
        "Zero Guest Bookings",
        "Negative Prices"
    ],
    "Before Cleaning": [
        len(df),
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        count_invalid(df),
        (df["adr"] < 0).sum()
    ],
    "After Cleaning": [
        len(df_clean),
        df_clean.isnull().sum().sum(),
        df_clean.duplicated().sum(),
        count_invalid(df_clean),
        (df_clean["adr"] < 0).sum()
    ]
})

display(summary)


### 6.8 Investigating Outliers

We will use the Interquartile Range (IQR) method to identify
unusually high or low values in hotel prices and booking
lead times.

We will investigate these outliers before deciding whether
to remove them.

In [ ]:
# Check outliers using the IQR method
for col in ["adr", "lead_time"]:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df_clean[
        (df_clean[col] < lower) |
        (df_clean[col] > upper)
    ]

    print("Column:", col)
    print("Lower Bound:", lower)
    print("Upper Bound:", upper)
    print("Number of Outliers:", len(outliers))
    print()


### 6.9 Final Cleaning Decisions

We handled missing values and removed bookings with invalid
guest counts or negative prices.

We kept duplicate rows because the dataset does not have
a unique booking ID, so identical rows may represent
different reservations.

We also kept the statistical outliers because unusually
high prices and lead times may be valid.

These remaining issues will be considered during
machine learning.

In [ ]:
# Save the cleaned dataset
df_clean.to_csv("hotel_bookings_cleaned.csv", index=False)

# Final verification
print("Cleaned Dataset Shape:", df_clean.shape)
print("Remaining Missing Values:", df_clean.isnull().sum().sum())
print("Remaining Duplicate Rows:", df_clean.duplicated().sum())

print("Cleaned dataset saved successfully!")
